In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

FILE_PATH = "multiclass_data.csv"

df = pd.read_csv(FILE_PATH)

print(df.head())
print(df.shape)
df.info()
print(df.describe(include="all").T)

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

TARGET = "target"

ID_COLUMNS = []

CATEGORICAL_COLUMNS = [
    # "category_1",
    # "category_2",
]

NUMERIC_COLUMNS = [
    # "feature_1",
    # "feature_2",
]

# Auto-detect when the lists above are empty
feature_columns = [
    c for c in df.columns
    if c not in [TARGET] + ID_COLUMNS
]

if not CATEGORICAL_COLUMNS:
    CATEGORICAL_COLUMNS = (
        df[feature_columns]
        .select_dtypes(include=["object", "category", "bool"])
        .columns.tolist()
    )

if not NUMERIC_COLUMNS:
    NUMERIC_COLUMNS = [
        c for c in feature_columns
        if c not in CATEGORICAL_COLUMNS
    ]

print("Categorical:", CATEGORICAL_COLUMNS)
print("Numeric:", NUMERIC_COLUMNS)

In [ ]:
# ============================================================
# 3. EDA
# ============================================================

print("\n========== TARGET DISTRIBUTION ==========")
print(df[TARGET].value_counts())
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

sns.countplot(data=df, x=TARGET)
plt.title("Target Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n========== MISSING VALUES ==========")
print(
    pd.DataFrame({
        "count": df.isnull().sum(),
        "percent": (df.isnull().mean() * 100).round(2),
    }).query("count > 0")
)

print("\nDuplicates:", df.duplicated().sum())

if NUMERIC_COLUMNS:
    corr = df[NUMERIC_COLUMNS].corr()
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Numeric Correlation")
    plt.tight_layout()
    plt.show()

for column in CATEGORICAL_COLUMNS:
    rates = pd.crosstab(
        df[column],
        df[TARGET],
        normalize="index"
    )
    rates.plot(kind="bar", stacked=True, figsize=(7, 4))
    plt.title(f"Class Distribution by {column}")
    plt.ylabel("Proportion")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 4. FEATURES / TARGET / LABEL ENCODING
# ============================================================

X = df.drop(columns=[TARGET] + ID_COLUMNS)
y = df[TARGET]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", list(label_encoder.classes_))
print("Encoded classes:", np.unique(y_encoded))

In [ ]:
# ============================================================
# 5. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# ============================================================
# 6. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_COLUMNS),
    ("cat", categorical_pipeline, CATEGORICAL_COLUMNS),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)

In [ ]:
# ============================================================
# 7. MODELS
# ============================================================

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    print(name, "trained")

In [ ]:
# ============================================================
# 8. EVALUATION
# ============================================================

def evaluate_model(model, X, y, model_name):
    predictions = model.predict(X)

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y, predictions),
        "precision_macro": precision_score(
            y, predictions, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y, predictions, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y, predictions, average="macro", zero_division=0
        ),
        "f1_weighted": f1_score(
            y, predictions, average="weighted", zero_division=0
        ),
    }

    print(f"\n========== {model_name} ==========")
    print(pd.Series(results))
    print("\nClassification Report:")
    print(
        classification_report(
            y,
            predictions,
            target_names=label_encoder.classes_.astype(str),
            zero_division=0,
        )
    )

    return results

results = []

for name, model in models.items():
    results.append(
        evaluate_model(model, X_val_processed, y_val, name)
    )

results_df = pd.DataFrame(results).sort_values(
    "f1_macro",
    ascending=False
)

print(results_df)

In [ ]:
# ============================================================
# 9. CONFUSION MATRICES
# ============================================================

for name, model in models.items():
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_val_processed,
        y_val,
        display_labels=label_encoder.classes_,
        cmap="Blues",
    )
    plt.title(name)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 10. FINAL TEST
# ============================================================

BEST_MODEL_NAME = results_df.iloc[0]["model"]
best_model = models[BEST_MODEL_NAME]

test_predictions = best_model.predict(X_test_processed)

print("Selected model:", BEST_MODEL_NAME)
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=label_encoder.classes_.astype(str),
        zero_division=0,
    )
)

print(
    "Macro F1:",
    f1_score(y_test, test_predictions, average="macro")
)